# Instalación e Importación de dependencias

Para la realización de este ejercicio se usa kagglehub para descargar el dataset, pandas, numpy, operator para trabajar con datos y estructuras de datos. Y sklearn y deap para la creación del modelo

In [1]:
!pip install deap update-checker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 22.4 MB/s eta 0:00:00


In [2]:
import kagglehub
import pandas as pd
import numpy as np
import os
import operator
import random

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

from deap import base, creator, tools, gp

# Descarga del dataset

Para este ejercicio se usa un dataset llamado [Fraud Detection](https://www.kaggle.com/datasets/waddahali/fraud-detection/data) el cual contiene varios registros de transacciones bancarias. Cada registro tiene la cantidad de la transacción, la hora del día, la edad del cliente, etc.

In [3]:
path = kagglehub.dataset_download("waddahali/fraud-detection")
print("Path to dataset files:", path)

PATH = path

100%|██████████| 321k/321k [00:00<00:00, 31.3MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/waddahali/fraud-detection/versions/1


# Carga y Procesamiento de Datos
Primero se define un tipo de función de aptitud (fitness) y de individuo. Luego de eso se carga el Dataset, se limpia el dataset quitando todos los N/A, se obtienen los tags que clasifican cada transacción como fraudulenta o no y finalmente se separan los conjuntos de entrenamiento y de prueba.

In [9]:
# Define un tipo de función de fitness (maximizar la precisión) y de individuo
try:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)
except AttributeError:
    pass

file_name = 'fraud.csv'
file_path = os.path.join(PATH, file_name)

try:
    df = pd.read_csv(file_path)
    print(f"Dataset '{file_name}' cargado exitosamente.")
except FileNotFoundError:
    print(f"Error: '{file_path}' no encontrado.")

df = df.fillna(df.mean(numeric_only=True))

if 'is_fraud' not in df.columns:
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
else:
    X = df.drop('is_fraud', axis=1)
    y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de y_train: {y_train.shape}")
print(f"Forma de X_test: {X_test.shape}")
print(f"Forma de y_test: {y_test.shape}")
print(f"Transacciones fraudulentas en el conjunto de entrenamiento: {y_train.sum()}/{len(y_train)} ({y_train.mean()*100:.2f}%) ")
print(f"Transacciones fraudulentas en el conjunto de prueba: {y_test.sum()}/{len(y_test)} ({y_test.mean()*100:.2f}%) ")

Dataset 'fraud.csv' cargado exitosamente.
Forma de X_train: (4900, 12)
Forma de y_train: (4900,)
Forma de X_test: (2100, 12)
Forma de y_test: (2100,)
Transacciones fraudulentas en el conjunto de entrenamiento: 505/4900 (10.31%) 
Transacciones fraudulentas en el conjunto de prueba: 216/2100 (10.29%) 


/usr/local/lib/python3.13/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.13/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


# Configuración para Programación Genética

Primero se definen las operaciones disponibles para el algoritmo, que son suma, resta, multiplicación, división y negación. Luego se crea la toolbox de Deap, que ayuda en la creación de arboles. Luego se define la función de detección, la cual evalua cada registro para determinar si hubo fraude o no. Finalmente, se define el método de selección (por torneo) y cruce, y la mutación de los árboles.

In [5]:
def protectedDiv(left, right):
    try:
        return left / right
    except ZeroDivisionError:
        return 1.0

pset = gp.PrimitiveSet("MAIN", X_train.shape[1])
pset.addPrimitive(operator.add, 2)
pset.addPrimitive(operator.sub, 2)
pset.addPrimitive(operator.mul, 2)
pset.addPrimitive(protectedDiv, 2)
pset.addPrimitive(operator.neg, 1)

for i, col in enumerate(X_train.columns):
    pset.renameArguments(**{f'ARG{i}': col})

toolbox = base.Toolbox()
# expr: Generador de expresiones, utilizado para crear árboles iniciales
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=2)
# individual: Crea un Individuo (un PrimitiveTree) a partir de una expresión
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
# population: Crea una lista de individuos
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
# compile: Compila el árbol de expresiones en una función ejecutable
toolbox.register("compile", gp.compile, pset=pset)

def evalFraudDetection(individual, X, y):
    func = toolbox.compile(expr=individual)
    predictions = []

    for i in range(X.shape[0]):
        try:
            result = func(*X.iloc[i].values)
            predictions.append(1 if result > 0.5 else 0)
        except OverflowError:
            predictions.append(0)
        except Exception as e:
            predictions.append(0)

    if not predictions:
        return 0.0,

    predictions = np.array(predictions)
    accuracy = accuracy_score(y, predictions)
    return accuracy, # El fitness de DEAP espera una tupla

toolbox.register("evaluate", evalFraudDetection, X=X_train, y=y_train)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("mate", gp.cxOnePoint)

# mutUniform: Muta un individuo reemplazando un subárbol elegido al azar
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr, pset=pset)

# Entrenamiento del Modelo

Primero se definen los parámetros del modelo. Luego se inicializa la población e inicia el proceso. Para cada etapa se seleccionan los individuos que generaran la siguiente población, luego se evalua esa población y se registran las estadísticas.

In [6]:
pop_size = 100  # Número de individuos en la población
ngen = 30       # Número de generaciones a evolucionar
cxpb = 0.7      # Probabilidad de cruce
mutpb = 0.2     # Probabilidad de mutación
indpb = 0.05    # Probabilidad para la generación de expresiones en la mutación

random.seed(42)
np.random.seed(42)

# Inicializa la población
pop = toolbox.population(n=pop_size)
hof = tools.HallOfFame(1) # Almacena el mejor individuo encontrado hasta ahora
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

print("Iniciando la evolución de Programación Genética...")
# Evalúa la población inicial
fitnesses = list(map(lambda ind: toolbox.evaluate(ind, X=X_train, y=y_train), pop))
for ind, fit in zip(pop, fitnesses):
    ind.fitness.values = fit

hof.update(pop) # Actualiza el Salón de la Fama con la población inicial

for gen in range(1, ngen + 1):
    print(f"-- Generación {gen} --")
    offspring = toolbox.select(pop, len(pop))       # Selecciona los individuos de la siguiente generación
    offspring = list(map(toolbox.clone, offspring)) # Clona los individuos seleccionados para evitar modificar directamente a los padres

    # Aplica cruce a la descendencia
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < cxpb:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    for mutant in offspring:
        if random.random() < mutpb:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    # Evalúa los individuos con un fitness inválido
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = map(lambda ind: toolbox.evaluate(ind, X=X_train, y=y_train), invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    # Reemplaza la población antigua por la descendencia
    pop[:] = offspring

    # Actualiza el Salón de la Fama
    hof.update(pop)

    # Registra estadísticas
    record = stats.compile(pop)
    print(f"Fitness Mínimo: {record['min']:.4f}, Fitness Máximo: {record['max']:.4f}, Fitness Promedio: {record['avg']:.4f}, Desv. Estándar: {record['std']:.4f}")

print("\n-- Evolución terminada --")

best_individual = hof[0]
print(f"\n--- Mejor Individuo (Árbol de Expresión) ---")
print(best_individual)
print(f"Fitness (Precisión en el conjunto de entrenamiento): {best_individual.fitness.values[0]:.4f}")

Iniciando la evolución de Programación Genética...


/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide 

-- Generación 1 --


/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by

Fitness Mínimo: 0.1051, Fitness Máximo: 0.8969, Fitness Promedio: 0.7672, Desv. Estándar: 0.2558
-- Generación 2 --


...

Fitness Mínimo: 0.1031, Fitness Máximo: 0.8969, Fitness Promedio: 0.8238, Desv. Estándar: 0.2132
-- Generación 29 --


/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide b

Fitness Mínimo: 0.1031, Fitness Máximo: 0.8971, Fitness Promedio: 0.8143, Desv. Estándar: 0.2199
-- Generación 30 --


/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return left / right
/tmp/ipykernel_531/3938121451.py:6: RuntimeWarning: divide

Fitness Mínimo: 0.1031, Fitness Máximo: 0.8971, Fitness Promedio: 0.8277, Desv. Estándar: 0.2130

-- Evolución terminada --

--- Mejor Individuo (Árbol de Expresión) ---
neg(add(protectedDiv(is_first_transaction, velocity_score), hour_of_day))
Fitness (Precisión en el conjunto de entrenamiento): 0.8971


# Evaluación del Modelo
Simplemente, sobre los datos de prueba, se evalua el modelo generando predicciones sobre cada caso. Al final, se muestra la precisión, la matriz de confución y su explicación.

In [10]:
print("\n--- Evaluando el Mejor Individuo en el Conjunto de Prueba ---")
test_accuracy = toolbox.evaluate(best_individual, X=X_test, y=y_test)[0]
print(f"Precisión en el Conjunto de Prueba: {test_accuracy:.4f}")

final_func = toolbox.compile(expr=best_individual)
test_predictions = []
for i in range(X_test.shape[0]):
    try:
        result = final_func(*X_test.iloc[i].values)
        test_predictions.append(1 if result > 0.5 else 0)
    except OverflowError:
        test_predictions.append(0)
    except Exception as e:
        test_predictions.append(0)

test_predictions = np.array(test_predictions)
cm = confusion_matrix(y_test, test_predictions)
print("\nMatriz de Confusión en el Conjunto de Prueba:")
print(cm)
print(f"\nVerdaderos Positivos (Fraude Correctamente Detectado): {cm[1, 1]}")
print(f"Falsos Positivos (No Fraude Detectado como Fraude): {cm[0, 1]}")
print(f"Verdaderos Negativos (No Fraude Correctamente Detectado): {cm[0, 0]}")
print(f"Falsos Negativos (Fraude No Detectado): {cm[1, 0]}")



--- Evaluando el Mejor Individuo en el Conjunto de Prueba ---
Precisión en el Conjunto de Prueba: 0.8962

Matriz de Confusión en el Conjunto de Prueba:
[[1882    2]
 [ 216    0]]

Verdaderos Positivos (Fraude Correctamente Detectado): 0
Falsos Positivos (No Fraude Detectado como Fraude): 2
Verdaderos Negativos (No Fraude Correctamente Detectado): 1882
Falsos Negativos (Fraude No Detectado): 216


# Caso de Prueba Inventado
Se realizan 2 casos de fraude, uno donde potencialmente hubo fraude y otro donde no. Para esto se promedian los valores del dataset para generar un caso probable y en el caso de fraude se exagera una variable.

In [8]:
feature_names = X_train.columns.tolist()

non_fraud_example_values = X_train.mean().to_dict()
non_fraud_df = pd.DataFrame([non_fraud_example_values], columns=feature_names)

fraud_example_values = X_train.mean().to_dict()
if len(feature_names) > 0:
    first_feature_name = feature_names[0]
    fraud_example_values[first_feature_name] = X_train[first_feature_name].max() * 2

fraud_df = pd.DataFrame([fraud_example_values], columns=feature_names)
final_func = toolbox.compile(expr=best_individual)


print("--- Caso de Prueba: Potencialmente NO FRAUDE ---")
non_fraud_prediction_raw = final_func(*non_fraud_df.iloc[0].values)
non_fraud_prediction_class = 1 if non_fraud_prediction_raw > 0.5 else 0

print(f"Valores del caso (promedio de características): {non_fraud_df.iloc[0].to_dict()}")
print(f"Predicción cruda del modelo: {non_fraud_prediction_raw:.4f}")
print(f"Clasificación: {'Fraude' if non_fraud_prediction_class == 1 else 'No Fraude'}")


print("\n--- Caso de Prueba: Potencialmente FRAUDE ---")
fraud_prediction_raw = final_func(*fraud_df.iloc[0].values)
fraud_prediction_class = 1 if fraud_prediction_raw > 0.5 else 0

print(f"Valores del caso (primera característica anómalamente alta): {fraud_df.iloc[0].to_dict()}")
print(f"Predicción cruda del modelo: {fraud_prediction_raw:.4f}")
print(f"Clasificación: {'Fraude' if fraud_prediction_class == 1 else 'No Fraude'}")

--- Caso de Prueba: Potencialmente NO FRAUDE ---
Valores del caso (promedio de características): {'transaction_amount': 100.02787025449528, 'hour_of_day': 2.1987404634034062, 'is_weekend': 0.10865829713809722, 'num_items': 2.9795977277508943, 'customer_age': 36.40995962217288, 'prev_transactions': 4.505840590614126, 'distance_from_home': 25.043217899309127, 'device_type': 0.6958279883381925, 'network_quality': 74.05692877267556, 'is_first_transaction': 0.09594592888701872, 'store_type': 0.28509246147438566, 'velocity_score': 4.984544157942164}
Predicción cruda del modelo: -2.2180
Clasificación: No Fraude

--- Caso de Prueba: Potencialmente FRAUDE ---
Valores del caso (primera característica anómalamente alta): {'transaction_amount': 592.6237706436328, 'hour_of_day': 2.1987404634034062, 'is_weekend': 0.10865829713809722, 'num_items': 2.9795977277508943, 'customer_age': 36.40995962217288, 'prev_transactions': 4.505840590614126, 'distance_from_home': 25.043217899309127, 'device_type': 0.6

El modelo clasifica erroneamente el caso de prueba de fraude. Esto se puede deber a que, al haber demasiados casos de prueba que no son fraude, prefiere siempre devolver como resultado que ningún caso es fraude.

Para solucionar esto se puede incrementar los casos de prueba de fraude artificialmente, darles más peso a los casos de fraude o quitarles peso a los casos que no son fraude.